In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [ ]:
para_list = ['LATP', 'LONP','RALT','GS','TAS','IVV','BLAC','CTAC','FPAC','LATG','N1_1','FLAP','PTCH','ROLL','AOA1',\
            'AIL_1','ELEV_1','RUDD','LOC','GLS','ALT','LATG','PTRM','LONG','OIT_1']

def load_data(read_url):
    print('Loading data ...')
    train_y_list = np.load(read_url + 'train_y_list.npy', allow_pickle=True)
    test_y_list = np.load(read_url + 'test_y_list.npy', allow_pickle=True)
    train_X_list = np.load(read_url + 'train_x_list.npy', allow_pickle=True)
    test_X_list = np.load(read_url + 'test_x_list.npy', allow_pickle=True)

    print("Data shapes: ")
    print(f"train_x_list: {train_X_list.shape}")
    print(f"training data within each fold: {train_X_list[0].shape}")
    print(f"train_y_list: {train_y_list.shape}")
    print(f"training label within each fold: {train_y_list[0].shape}")

    return train_y_list, test_y_list, train_X_list, test_X_list

def reshape_data(data):
    # Ensure data is a 3D array: (samples, timesteps, features)
    num_samples, num_timesteps, num_features = data.shape

    # Flatten the data to 2D for standardization
    flattened_data = data.reshape(num_samples * num_timesteps, num_features)
    return flattened_data, (num_samples, num_timesteps, num_features)

def unflatten_data(flat_data, original_shape):
    num_samples, num_timesteps, num_features = original_shape
    return flat_data.reshape(num_samples, num_timesteps, num_features)

def preprocess_data(train_X_list, test_X_list):
    print('Preprocessing data ...')
    # Find indices of 'TAS' and 'GS'
    excluded_tags = ['TAS', 'GS']
    excluded_index = [para_list.index(tag) for tag in excluded_tags if tag in para_list]

    # Calculate indices to keep
    keep_index = np.sort(list(set(np.arange(train_X_list[0].shape[-1])) - set(excluded_index)))

    # Filter the data
    train_x_list_filtered = [data[:, :, keep_index] for data in train_X_list]
    test_x_list_filtered = [data[:, :, keep_index] for data in test_X_list]

    # Initialize the scaler
    scaler = StandardScaler()

    # Standardize each 3D array in the list
    for i in range(len(train_X_list)):  # Adjust this range based on the actual length of your lists
        # Reshape train and test data
        flattened_train_data, train_shape = reshape_data(train_x_list_filtered[i])
        flattened_test_data, test_shape = reshape_data(test_x_list_filtered[i])

        # Fit scaler on train data and transform both train and test data
        scaler.fit(flattened_train_data)
        train_standardized = scaler.fit_transform(flattened_train_data)
        test_standardized = scaler.transform(flattened_test_data)

        # Unflatten data back to 3D
        train_x_list_filtered[i] = unflatten_data(train_standardized, train_shape)
        test_x_list_filtered[i] = unflatten_data(test_standardized, test_shape)

    return train_x_list_filtered, test_x_list_filtered

In [ ]:
def truncate_data(data: np.ndarray) -> np.ndarray:
    """
    Truncate the input NumPy array to keep only the first 60 time steps.

    The input array is assumed to have a shape of
    (number of records, time steps, features). For instance, if the input
    is of shape (100, 81, 23), this function will return an array of shape
    (100, 60, 23) by selecting the first 60 time steps along the second dimension.

    Parameters:
        data (np.ndarray): The input data array with shape (records, time_steps, features).

    Returns:
        np.ndarray: The truncated array with shape (records, 60, features).

    Raises:
        ValueError: If the input array does not have at least 60 time steps.
    """
    if data.shape[1] < 60:
        raise ValueError("The input data must have at least 60 time steps.")

    # Slice the array to keep all records, first 60 time steps, and all features.
    return data[:, :60, :]

In [ ]:
def truncate_pad_data(data, percentage_map, original_length, dtype=np.float32):
    """
    Truncate the dataset based on specified percentages and their proportions, then pad back to original length.

    Parameters:
    - data (np.array): The original dataset.
    - percentage_map (dict): A dictionary where keys are tuples representing the percentage intervals of the original length to keep,
                             and values are the proportion of the dataset to be truncated to this length.
    - original_length (int): The original length of the sequences.
    - dtype (type): Desired data type for the final dataset (default: np.float32).

    Returns:
    - np.array: The truncated and padded dataset with consistent dtype and padding.
    """
    total_samples = len(data)
    truncated_padded_datasets = []

    start_index = 0
    for interval, proportion in percentage_map.items():
        min_percentage, max_percentage = interval
        num_samples = int(total_samples * proportion)
        end_index = start_index + num_samples

        # Slice the data segment
        data_segment = data[start_index:end_index]

        # Truncate each sequence in the segment
        truncated_segment = []
        for sequence in data_segment:
            random_percentage = np.random.randint(min_percentage, max_percentage + 1)
            truncated_length = max(1, int(original_length * (random_percentage / 100)))

            truncated_sequence = sequence[:truncated_length]
            padded_sequence = np.pad(
                truncated_sequence,
                ((0, original_length - truncated_length), (0, 0)),
                mode='constant',
                constant_values=0.0
            ).astype(dtype)  # Force consistent data type

            truncated_segment.append(padded_sequence)

        truncated_segment = np.array(truncated_segment, dtype=dtype)
        truncated_padded_datasets.append(truncated_segment)

        start_index = end_index

    # Concatenate all truncated and padded segments
    final_truncated_padded_data = np.concatenate(truncated_padded_datasets, axis=0)
    return final_truncated_padded_data.astype(dtype)  # Ensure consistent dtype

def shuffle_data(train_x_list, train_y_list):
    """
    Shuffle the training data and labels in the same order.

    Parameters:
    - train_x_list (np.array): The dataset of input features.
    - train_y_list (np.array): The dataset of labels.

    Returns:
    - np.array: The shuffled dataset of input features.
    - np.array: The shuffled dataset of labels.
    """
    # Generate random indices for shuffling

    indices = np.arange(len(train_x_list))
    np.random.shuffle(indices)

    # Apply the shuffled indices to both train_x_list and train_y_list
    shuffled_x_list = train_x_list[indices]
    shuffled_y_list = train_y_list[indices]

    return shuffled_x_list, shuffled_y_list